# BrainAI on Colab (GPU)
Runtime > Change runtime type > GPU. Data and checkpoints live on Drive so sessions can resume.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!git clone https://github.com/ZahidServers/BrainAI.git /content/BrainAI
%cd /content/BrainAI
!pip -q install torch datasets pillow opencv-python-headless transformers accelerate
import os; D='/content/drive/MyDrive/brainai'; os.makedirs(D+'/data',exist_ok=True)
!ln -sfn $D/data /content/BrainAI/data_big

## 1. Text corpora (reading/writing)

In [ ]:
!python tools/fetch_text.py --out $D/data --mb 500
!ln -sf $D/data/*.txt data/   # corpus.py auto-discovers every data/*.txt as a domain
!gunzip -kc data_compressed/software_engineering_105000_code_enabled_txt_corpus.txt.gz > data/software_engineering_105000_code_enabled_txt_corpus.txt   # kept gzipped in git (GitHub 100 MB limit)

## 2. Sensor data (images, audio, video)

In [ ]:
!python tools/fetch_sensors.py --out $D/sensors --n 20000
!ln -sfn $D/sensors sensors

## 2b. Robot data (state, actions, camera) - LeRobot format

In [ ]:
!python tools/fetch_robot.py --out $D/robot_data --skip koch_pick_place_1_lego
!python tools/fetch_robot.py --out $D/robot_data --humanoid          # Unitree H1, ~1.4 GB
# optional, 9-18 GB each: !python tools/fetch_robot.py --out $D/robot_data --humanoid --humanoid-big
!ln -sfn $D/robot_data robot_data

## 2c. Train the perception encoders (held-out accuracy is printed vs chance)

In [ ]:
for m,s in [('images',4000),('audio',3000),('video',2000)]:
    !python train_sensors.py --modality $m --steps $s --batch 64 --scale small

## 2d. Behaviour cloning (state -> action); compares to copy-state baseline

In [ ]:
import glob
for d in sorted(glob.glob('robot_data/*')):
    !python train_robot.py --dataset $d --steps 3000 --scale small

## 3. Distil Qwen into BrainAI (text; teacher writes, student trains)
Byte tokenizer != Qwen BPE, so logit-KL is impossible; we use sequence distillation.

In [ ]:
!python distill.py generate --teacher Qwen/Qwen2.5-1.5B-Instruct --samples 50000
!cp data/teacher_corpus.jsonl $D/

## 4. Train (text + distilled). Start with `small`, move to `base` once loss curves look sane.

In [ ]:
!python train_multidomain.py --scale small --steps 20000 --batch 32 --seq 256

In [ ]:
!cp -r checkpoints $D/checkpoints